# 04 · BIXI Model Results and Interpretation

**A concise results report for the BIXI Smart Availability research prototype.**

The goal is to forecast arrivals/departures at BIXI stations and ultimately answer:
**Will there be at least one bike or free dock when I arrive?**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelcolor": "#293847",
    "axes.titlesize": 12,
    "font.size": 10,
})

GREEN, BLUE, GRAY, RED = "#26825E", "#407BB0", "#9EABB8", "#C1584B"

## 1. Historical demand forecasting: generalization across years

Here, **MAE** measures the average absolute number of bikes by which a 15-minute arrival
or departure count prediction differs from the recorded trip count. Lower is better.


In [ ]:
# Transcribed from the matched 2025–2026 experiment; not recomputed in this notebook.
demand = pd.DataFrame([
    (2025, "Arrivals", 8_817_216, .6541, .7289, 10.2602),
    (2025, "Departures", 8_817_216, .6599, .7320, 9.8456),
    (2025, "Net flow", 8_817_216, .7311, .7064, -3.5092),
    (2026, "Arrivals", 8_817_216, .6499, .7059, 7.9254),
    (2026, "Departures", 8_817_216, .6553, .7080, 7.4415),
    (2026, "Net flow", 8_817_216, .7207, .6952, -3.6764),
], columns=["year", "target", "intervals", "model_mae", "zero_baseline_mae", "improvement_pct"])
# Improvement was calculated in the original experiment *before* MAE rounding.
display(demand.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, year in zip(axes, (2025, 2026)):
    values = demand[(demand.year == year) & demand.target.isin(["Arrivals", "Departures"])]
    positions = np.arange(len(values))
    ax.bar(positions - .19, values.zero_baseline_mae, .38, color=GRAY,
           label="Predict zero")
    ax.bar(positions + .19, values.model_mae, .38, color=GREEN,
           label="CatBoost")
    for i, pct in enumerate(values.improvement_pct):
        ax.text(i, .77, f"{pct:.1f}% less error", ha="center", color=GREEN,
                fontweight="bold", fontsize=9)
    ax.set_xticks(positions, values.target)
    ax.set_ylim(0, .85)
    ax.set_title(f"{year} — unseen data")
    ax.grid(axis="y", alpha=.15)
axes[0].set_ylabel("Mean absolute error (trips / 15 min)")
axes[0].legend(loc="upper left", frameon=False)
fig.suptitle("Historical demand forecasting: CatBoost versus zero baseline", y=1.03)
fig.tight_layout()
plt.show()

## 2. Forecast bias: what is systematically missed?

Positive bias means overprediction, negative bias means underprediction. The means below
refer to the matched 2025–2026 evaluation cohort, not all BIXI stations or dates.

In [ ]:
bias = pd.DataFrame([
    (2025, "Arrivals", .7289, .6935, -.0355),
    (2025, "Departures", .7320, .7007, -.0313),
    (2026, "Arrivals", .7059, .6931, -.0128),
    (2026, "Departures", .7080, .7003, -.0077),
], columns=["year", "target", "actual_mean", "predicted_mean", "bias"])
# The original experiment calculated bias before rounding displayed means.
display(bias.round(4))

fig, ax = plt.subplots(figsize=(8.5, 3.8))
bars = ax.bar(
    [f"{r.year} {r.target}" for r in bias.itertuples()],
    bias.bias, color=[BLUE if y == 2025 else GREEN for y in bias.year]
)
ax.axhline(0, lw=1, color="#3D4A55")
ax.set_ylabel("Predicted mean − actual mean (trips / 15 min)")
ax.set_title("Both test years show slight underprediction")
for bar, value in zip(bars, bias.bias):
    ax.annotate(f"{value:+.4f}",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, -16), textcoords="offset points", ha="center", color="white",
                fontsize=8)
ax.grid(axis="y", alpha=.15)
fig.tight_layout()
plt.show()

## 3. Arrival-time availability: probability quality

The operational question differs from trip-count forecasting:

- For a rider returning a bike: **P(at least one free dock in 15 minutes)**.
- For a rider departing: **P(at least one rentable bike in 15 minutes)**.

The initial experiment combined the demand forecasts with current GBFS inventory and a
Skellam approximation of net arrivals/departures. **Brier score** is the mean squared error
of predicted probabilities against observed binary availability; **lower is better**.

The persistence baseline assumed the current available/unavailable state stays unchanged.

In [ ]:
# Reported pilot outcomes only: ~35-minute data collection, 2,412 matched examples.
availability = pd.DataFrame([
    ("Bike", 2412, 97.8856, 97.8856, .0211, .0165),
    ("Dock", 2412, 98.5075, 98.5075, .0149, .0109),
], columns=["resource", "examples", "baseline_accuracy_pct", "prototype_accuracy_pct",
            "baseline_brier", "prototype_brier"])
availability["brier_reduction_pct"] = 100 * (
    availability.baseline_brier - availability.prototype_brier
) / availability.baseline_brier
display(availability.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4.4))
positions = np.arange(len(availability))
ax.bar(positions - .19, availability.baseline_brier, .38, color=GRAY,
       label="Persistence baseline")
ax.bar(positions + .19, availability.prototype_brier, .38, color=BLUE,
       label="Probability prototype")
for i, pct in enumerate(availability.brier_reduction_pct):
    ax.text(i, .024, f"{pct:.1f}% lower Brier", color=BLUE, ha="center",
            fontsize=10, fontweight="bold")
ax.set_xticks(positions, ["At least one bike", "At least one free dock"])
ax.set_ylim(0, .028)
ax.set_ylabel("Brier score (lower is better)")
ax.set_title("Preliminary 15-minute probability quality")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=.15)
fig.tight_layout()
plt.show()

## 4. Important edge cases: nearly empty or full stations

These are the stations that matter most for the envisioned app. We specifically look at
observations beginning with only **one or two** bikes or docks available.

In [ ]:
critical = pd.DataFrame([
    ("1–2 bikes", 315, .0540, .0609),
    ("1–2 free docks", 139, .1511, .1144),
], columns=["starting_inventory", "examples", "baseline_brier", "prototype_brier"])
critical["relative_improvement_pct"] = 100 * (
    critical.baseline_brier - critical.prototype_brier
) / critical.baseline_brier
display(critical.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4))
x = np.arange(len(critical))
ax.bar(x - .19, critical.baseline_brier, .38, label="Persistence", color=GRAY)
ax.bar(x + .19, critical.prototype_brier, .38, label="Prototype", color=BLUE)
ax.set_xticks(x, [f"{r.starting_inventory}\n(n={r.examples})" for r in critical.itertuples()])
ax.set_ylabel("Brier score (lower is better)")
ax.set_title("Mixed results in low-inventory cases")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=.15)
fig.tight_layout()
plt.show()